# Feature Engineering

In [17]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [14]:
DATA_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data"

df = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "heart_failure_clinical_records_dataset.csv"
    )
)

X = df.drop("DEATH_EVENT", axis=1)
y = df["DEATH_EVENT"]

In [15]:
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (239, 12)
Test shape : (60, 12)


In [19]:
def add_egfr(df_new):
    df_new = df_new.copy()
    scr, age, sex = df_new["serum_creatinine"], df_new["age"], df_new["sex"]
    k = np.where(sex == 0, 0.7, 0.9)
    alpha = np.where(sex == 0, -0.241, -0.302)
    df_new["egfr"] = (142 * np.minimum(scr / k, 1.0) ** alpha * np.maximum(scr / k, 1.0) ** -1.200 * 0.9938 ** age)
    return df_new

def add_ef_group(df_new):
    df_new = df_new.copy()
    df_new["ef_group"] = np.where(df_new["ejection_fraction"] < 40, 0, np.where(df_new["ejection_fraction"] <= 49, 1, 2))
    return df_new

In [20]:
X_train_fe = add_ef_group(add_egfr(X_train))

print("eGFR range:", X_train_fe["egfr"].min().round(1), "-", X_train_fe["egfr"].max().round(1))
print("\nef_group distribution:")
print(X_train_fe["ef_group"].value_counts().sort_index())

eGFR range: 4.7 - 118.0

ef_group distribution:
ef_group
0    144
1     47
2     48
Name: count, dtype: int64


In [21]:
df_fe[["age", "serum_creatinine", "sex", "ejection_fraction", "egfr", "ef_group"]].head()

,age,serum_creatinine,sex,ejection_fraction,egfr,ef_group
0,75.0,1.9,1,20,36.332850,0
1,55.0,1.1,1,38,79.278006,0
2,65.0,1.3,1,20,60.965177,0
3,50.0,1.9,1,20,42.444816,0
4,65.0,2.7,0,20,18.758754,0


## Feature Engineering Evaluation